# Why do the EU report and the site reports show different numbers?

Hull and Nottingham, three tables: **Daily_Losses**, **Time_Utilisation**, **Plan_OPT**.

Everything starts from the **original** Provideam exports (`sharepoint/source_original`), the
same files for both pipelines. Each pipeline is the Power Query (M) of its own template, one
function per M step, applied in order. After each step the notebook records the row count and
the sum of every measure column. The step where the two sums split is a cause.

**Read-only.** Nothing is written. Run top to bottom, or change the settings and rerun.

Sources of the M steps: `Temp EU Daily OEE Report 2025 V1 1.pbit` (EU),
`Hull Site OEE Report 2025 V3.1 (2).pbit`, `Nottingham Site OEE Report 2025 V1 (8).pbit`.
Analysis: `prompt-response/site-dashboards-analysis.md` (sections 1.3 and 4a).

**Not modelled** (no effect on rows or measures): `Product Key`, `Targets Key` text columns other
than as lookup keys, shift colour split (Hull), `Breakdown Types`, `Loss Final Level`, hours columns,
`Week`, `Year`, type changes. **Assumed**: dates are day/month/year (en-GB), empty text equals
null, the `Total:` row of each export is dropped (the machine filters remove it in Power BI too),
unpivot drops empty glidepath cells.

## 1. Settings

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

# The notebook lives in prompt-response/, the repo root is one level up.
REPO = Path('..').resolve()
ORIG = REPO / 'sharepoint' / 'source_original'
REF = REPO / 'sharepoint' / 'reference'
DOWN = REPO / 'downloads'

# What to compare
SITES = ['Hull', 'Nottingham']
TABLES = ['Daily_Losses', 'Time_Utilisation', 'Plan_OPT']
MONTH = '2026-09'      # 'YYYY-MM' or None (all). Filters the main table only, not the lookups
MACHINES = None        # for example ['B2 Auto'] or None (all). Filters the main table only

# Which files does the report read? (the PBIX reads every file in the SharePoint folder)
#   'folder'     every file in the local original folder, in file name order (default)
#   'manifest'   only the files in MANIFEST, with their real "Date created"
#   'include'    only the names in INCLUDE[(site, kind)]
#   'exclude'    everything except the names in EXCLUDE[(site, kind)]
#   'no_overlap' drop a file when all its keys are in a bigger file (a guess, not the PBIX)
FILE_MODE = 'folder'
# CSV with columns: site, kind, file, created.  kind = plan_opt | line_losses | volume_loss_correction
# created = SharePoint "Date created" (any date format pandas reads)
MANIFEST = DOWN / 'file_manifest.csv'
INCLUDE = {}   # {('Hull', 'plan_opt'): ['file1.csv', ...]}
EXCLUDE = {}

# Optional site files that are not downloaded yet (None = use the regional Shift Naming file)
SITE_SHIFT_FILE = {'Nottingham': None, 'Hull': None}
# Site glidepath files (downloads/)
SITE_GLIDE = {
    'Hull': 'Glide Path Targets V7 Add Updates as New Columns.csv',
    'Nottingham': 'Notts GlidePath Targets Add Updates as New Columns.csv',
}

# The rules that differ between the reports. Each one is a switch.
#   file_order       name | created_desc      order of files before "Removed Duplicates" (Hull site sorts
#                                              by Date created, newest first; needs the manifest)
#   machine_filter   eu | none                the machine / area filters that only the EU report has
#   minutes          whole | seconds          EU drops seconds of Duration and plan times, site keeps them
#   sub_group_names  eu | site                Hull only: EU renames the loss sub groups, Hull site keeps raw
#   target_key       unique | machine         key of the targets: machine + "-" + site, or machine
#   glidepath        regional | site          which glidepath file feeds the targets
#   shift_file       regional | site | none   shift lookup (Hull site has none)
RULES = {
    'EU': {
        'file_order': 'name', 'machine_filter': 'eu', 'minutes': 'whole',
        'sub_group_names': 'eu', 'target_key': 'unique', 'glidepath': 'regional',
        'shift_file': 'regional',
    },
    'Hull': {
        'file_order': 'created_desc', 'machine_filter': 'none', 'minutes': 'seconds',
        'sub_group_names': 'site', 'target_key': 'machine', 'glidepath': 'site',
        'shift_file': 'none',
    },
    'Nottingham': {
        'file_order': 'name', 'machine_filter': 'none', 'minutes': 'seconds',
        'sub_group_names': 'eu', 'target_key': 'machine', 'glidepath': 'site',
        'shift_file': 'site',
    },
}

# Switch single M steps off (by name, see the step lists below), then rerun. 'EU' and 'SITE'.
OFF = {'EU': set(), 'SITE': set()}

# Machine filters copied from the EU queries
HULL_EU_EXCLUDE = [
    '4 Tonne', '6 Tonne', '6000L', 'Analgesics Processing', 'Analgesics X Press',
    'Bosch Capsulator ', 'Dettol 1', 'Evolution', 'Fitten', 'Gaviscon TPR',
    'Gaviscon XP700(A)', 'GG4', 'GG5', 'GG6', 'GG7', 'Lemsip Process',
    'Senokot Press', 'Symex suite', 'Vacumix',
]
NOTT_PLAN_AREAS = [' Strepsils Packing', 'SD Packing']
NOTT_PLAN_EXCLUDE = ['S14']
NOTT_TU_KEEP = [
    'N1', 'N10', 'N11', 'N12', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9 Auto', 'O5', 'O6',
    'S04', 'S05', 'S06', 'S08', 'S10', 'S11', 'S12', 'S13', 'T2', 'T7',
]
NOTT_LOSS_EXCLUDE = [
    'Bosch', 'Fette 23', 'Fette12', 'Fette2', 'Fette20', 'Fette4', 'Fette5', 'Gran0', 'S14',
]
LOSS_GROUPS = ['Not Logged', 'Planned Loss', 'Unplanned Loss']
LOSS_MACHINES_OUT = ['B1 - Auto', 'B4 Auto', 'B5 Auto']
SITE_MACHINE_OUT = {'Hull': 'GT4 - Auto', 'Nottingham': 'N11 Auto'}

MEASURES = {
    'Plan_OPT': ['Good Parts', 'Capacity', 'Downtime Units', 'Total Time', 'Planned OpT',
                 'OEE Time', 'UPDT Target', 'PDT Target', 'OEE Target'],
    'Time_Utilisation': ['OEE Time Total', 'Planned OPT Total', 'Total Time Total'],
    'Daily_Losses': ['Duration', 'Loss Units', 'Time in Mins', 'Loss in CUs', 'Value',
                     'Volume Total'],
}
KEYS = {
    'Plan_OPT': ['Date', 'MachineName', 'ShiftName', 'PartName', 'ShiftStartTime'],
    'Time_Utilisation': ['Date', 'MachineName'],
    'Daily_Losses': ['MachineName', 'StartTime'],
}
BREAKDOWN_BY = ['MachineName', 'Dept', 'Area', 'OEELossLvl3Group', 'OEELossLvl3SubGroup', 'Date']
TOP = 15   # rows shown per breakdown
TOL = 1e-6

## 2. Loading helpers (original files, lookups)

In [ ]:
class Ctx:
    """State of one pipeline run: site, rules and the warnings found on the way."""

    def __init__(self, site, rules):
        self.site = site
        self.rules = rules
        self.warns = []
        self.note = ''

    def warn(self, text):
        if text not in self.warns:
            self.warns.append(text)


def num(s):
    return pd.to_numeric(s.str.replace(',', '', regex=False), errors='coerce')


def mins(s):
    p = s.str.extract(r'^\s*(\d+):(\d+):(\d+)')
    return (pd.to_numeric(p[0]) * 60 + pd.to_numeric(p[1]) + pd.to_numeric(p[2]) / 60)


def parse_dt(s, fmt):
    out = pd.to_datetime(s, format=fmt, errors='coerce')
    miss = out.isna() & s.ne('')
    if miss.any():
        out[miss] = pd.to_datetime(s[miss], dayfirst=True, errors='coerce')
    return out


def text_date(d):
    return d.dt.strftime('%d/%m/%Y').fillna('')


_RAW = {}


def read_raw(path):
    """One original export. UTF-16 tab (Provideam) or UTF-8 csv, all text."""
    if path in _RAW:
        return _RAW[path]
    head = open(path, 'rb').read(2)
    if head in (b'\xff\xfe', b'\xfe\xff'):
        df = pd.read_csv(path, encoding='utf-16', sep='\t', dtype=str, keep_default_na=False)
    else:
        df = pd.read_csv(path, encoding='utf-8-sig', sep=None, engine='python', dtype=str,
                         keep_default_na=False)
    df = df.loc[:, [c for c in df.columns if c.strip() and not c.startswith('Unnamed')]]
    df = df[~df.iloc[:, 0].str.startswith('Total')].copy()
    df['_file'] = path.name
    _RAW[path] = df
    return df


NO_OVERLAP_KEYS = {
    'plan_opt': ['YYYY-MM-DD', 'MachineName', 'ShiftName', 'PartName', 'ShiftStartTime'],
    'line_losses': ['MachineName', 'StartTime'],
    'volume_loss_correction': ['ShiftStartDate', 'MachineName', 'OEELossLvl3Group',
                               'OEELossLvl3SubGroup', 'OEELossLvl3Desc'],
}


def list_files(site, kind):
    """[(path, created)] of the files the report reads, per FILE_MODE."""
    folder = ORIG / kind / f'site={site}'
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() == '.csv')
    created = {}
    if FILE_MODE == 'manifest':
        m = pd.read_csv(MANIFEST, dtype=str)
        m = m[(m['site'] == site) & (m['kind'] == kind)]
        created = dict(zip(m['file'], pd.to_datetime(m['created'], dayfirst=True)))
        files = [p for p in files if p.name in created]
    elif FILE_MODE == 'include':
        files = [p for p in files if p.name in INCLUDE.get((site, kind), [])]
    elif FILE_MODE == 'exclude':
        files = [p for p in files if p.name not in EXCLUDE.get((site, kind), [])]
    elif FILE_MODE == 'no_overlap':
        keys = NO_OVERLAP_KEYS[kind]
        sets = {p: set(map(tuple, read_raw(p)[keys].values)) for p in files}
        keep = []
        for p in files:
            bigger = [q for q in files if q != p and len(sets[q]) > len(sets[p])
                      and sets[p] <= sets[q]]
            if not bigger:
                keep.append(p)
        files = keep
    return [(p, created.get(p.name, pd.NaT)) for p in files]


def get_files(site, kind, order, ctx):
    files = list_files(site, kind)
    if order == 'created_desc':
        if files and all(pd.notna(c) for _, c in files):
            return [p for p, _ in sorted(files, key=lambda x: x[1], reverse=True)]
        ctx.warn(f'{site} {kind}: no "Date created" (manifest missing): file name order, '
                 'newest name first, is an ASSUMPTION')
        return [p for p, _ in sorted(files, key=lambda x: x[0].name, reverse=True)]
    return [p for p, _ in sorted(files, key=lambda x: x[0].name)]


_CAT = {}


def concat_files(site, kind, order, ctx):
    key = (site, kind, order, FILE_MODE, tuple(sorted(INCLUDE.items(), key=str)))
    if key not in _CAT:
        frames = [read_raw(p) for p in get_files(site, kind, order, ctx)]
        _CAT[key] = pd.concat(frames, ignore_index=True)
    return _CAT[key].copy()


def left_join(df, key, lk, lkey, cols, ctx, name):
    """M NestedJoin LeftOuter + Expand: a lookup key that repeats multiplies the rows."""
    lk = lk[[lkey] + cols].rename(columns={lkey: '_k'})
    dup = int(lk.duplicated('_k', keep=False).sum())
    if dup:
        ctx.warn(f'{name}: {dup} lookup rows share a key, the join multiplies rows')
    df = df.copy()
    df['_k'] = key
    out = df.merge(lk, how='left', on='_k')
    return out.drop(columns='_k')


def machine_key(df, col, ctx):
    if ctx.rules['target_key'] == 'unique':
        return df[col] + '-' + ctx.site
    return df[col]


def load_machine_names(site):
    p = next((REF / 'machine_names' / f'site={site}').glob('*.csv'))
    mn = pd.read_csv(p, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    return mn[['MachineName', 'Dept', 'Area', 'Product Type']]


def load_shift(ctx):
    mode = ctx.rules['shift_file']
    path = SITE_SHIFT_FILE.get(ctx.site)
    if mode == 'site' and path:
        sh = pd.read_csv(path, encoding='utf-8-sig', dtype=str, keep_default_na=False)
        sh = sh.rename(columns={sh.columns[0]: 'Shift Type'})
        return sh[['Shift Type', 'Shift']]
    if mode == 'site':
        ctx.warn(f'{ctx.site}: site Shift Pattern Naming file not given, regional file used')
    p = next((REF / 'shift_naming' / 'site=All').glob('*.csv'))
    sh = pd.read_csv(p, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    return sh.rename(columns={'ShiftName': 'Shift Type'})[['Shift Type', 'Shift']]


def load_glide(ctx):
    if ctx.rules['glidepath'] == 'regional':
        p = next((REF / 'glidepaths' / f'site={ctx.site}').glob('*.csv'))
    else:
        p = DOWN / SITE_GLIDE[ctx.site]
    g = pd.read_csv(p, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    ids = [c for c in ['Site Name', 'Line', 'OEE Bucket'] if c in g.columns]
    long = g.melt(id_vars=ids, var_name='Attribute', value_name='Value')
    long['Value'] = pd.to_numeric(long['Value'].str.replace(',', '', regex=False),
                                  errors='coerce')
    long = long[long['Value'].notna()].copy()
    if ctx.rules['target_key'] == 'unique':
        site_col = long['Site Name'] if 'Site Name' in long.columns else ctx.site
        long['_mk'] = long['Line'] + '-' + site_col
    else:
        long['_mk'] = long['Line']
    return long


def load_equipment(site):
    p = next((REF / 'equipment_coding' / f'site={site}').glob('*.csv'))
    eq = pd.read_csv(p, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    eq['Equipment'] = eq['Equipment'].replace('', 'Unclassed')
    return eq[['Merged Unique Identifier', 'Equipment', 'Lvl 3 Loss Name']]


def load_vlc(ctx):
    """Volume Loss Correction query (same steps in both reports, per site)."""
    df = concat_files(ctx.site, 'volume_loss_correction', 'name', ctx)
    for a, b in [('Breakdown - Mechanical', 'Breakdown'), ('Breakdown - Electrical', 'Breakdown'),
                 ('Lack of resources', 'Lack of Resource')]:
        df['OEELossLvl3SubGroup'] = df['OEELossLvl3SubGroup'].str.replace(a, b, regex=False)
    if ctx.site == 'Hull':
        df['OEELossLvl3SubGroup'] = df['OEELossLvl3SubGroup'].str.replace(
            'In Process Quality Control', 'In process Quality Control', regex=False)
    else:
        for a, b in [('Planned Losses', 'Planned Loss'), ('Unplanned Losses', 'Unplanned Loss')]:
            df['OEELossLvl3Group'] = df['OEELossLvl3Group'].str.replace(a, b, regex=False)
    day = parse_dt(df['ShiftStartDate'], '%d/%m/%Y %H:%M:%S').dt.normalize()
    df = df[df['MachineName'] != ''].copy()
    day = day.loc[df.index]
    parts = [text_date(day), df['MachineName'], df['OEELossLvl3Group'],
             df['OEELossLvl3SubGroup'], df['OEELossLvl3Desc']]
    df['Volume Key'] = parts[0].str.cat(parts[1:], sep=' ')
    df['Loss Units'] = np.floor(num(df['Loss Units']) + 0.5)
    df['Time in Mins'] = mins(df['Time'])
    return df[['Volume Key', 'Loss Units', 'Time in Mins']]


def load_targets_volume(ctx):
    """Plan OPT grouped by day and machine: sum of Capacity."""
    df = concat_files(ctx.site, 'plan_opt', 'name', ctx)
    df = df[(df['MachineName'] != '') & (df['MachineName'] != SITE_MACHINE_OUT[ctx.site])]
    day = parse_dt(df['YYYY-MM-DD'], '%Y-%m-%d')
    g = pd.DataFrame({'day': day, 'm': df['MachineName'], 'cap': num(df['Capacity'])})
    g = g.groupby(['day', 'm'], dropna=False)['cap'].sum(min_count=1).reset_index()
    g['Volume Key'] = g['m'] + ' ' + text_date(g['day'])
    return g.rename(columns={'cap': 'Volume Total'})[['Volume Key', 'Volume Total']]

## 3. Check first: do the original files overlap?

Power BI appends every file in the folder. If two files hold the same rows, both reports count
them twice, so this does not split EU and site, but it inflates both, and for the Hull losses
the file order decides which copy survives "Removed Duplicates". A key that appears in more than
one file is listed here. Plan OPT key: day, machine, shift, part, shift start. Losses key:
machine, start time.

In [ ]:
def overlap_check(sites=SITES):
    summary, pairs = [], {}
    ctx = Ctx('-', {})
    for site in sites:
        for kind, keys in NO_OVERLAP_KEYS.items():
            if kind == 'volume_loss_correction':
                continue
            files = [p for p, _ in list_files(site, kind)]
            if not files:
                continue
            df = pd.concat([read_raw(p) for p in files], ignore_index=True)
            multi = df.groupby(keys)['_file'].agg(lambda s: tuple(sorted(set(s))))
            cross = multi[multi.map(len) > 1]
            conflict = 0
            if len(cross):
                vals = [c for c in df.columns if c not in keys + ['_file']]
                sub = df.merge(cross.rename('files').reset_index()[keys], on=keys)
                conflict = int((sub.drop_duplicates(keys + vals).groupby(keys).size() > 1).sum())
            summary.append({
                'site': site, 'kind': kind, 'files': len(files), 'rows': len(df),
                'keys in >1 file': len(cross), 'of which values differ': conflict,
                'rows with a repeated key (any file)': int(df.duplicated(keys, keep=False).sum()),
            })
            pairs[(site, kind)] = cross.value_counts().head(10)
    display(pd.DataFrame(summary))
    for k, v in pairs.items():
        if len(v):
            print('\nMost frequent file sets sharing keys:', k)
            display(v.rename('keys').to_frame())


overlap_check()

## 4. The pipelines

One function per M step, named like the step. The list of steps is the same for the EU report
and the site report; the rules (settings) decide whether a step changes anything for that
report. A step that a report does not have is a no-op for it (the trace shows `skipped`).

In [ ]:
def S_all_files(df, ctx, kind):
    # Only the Line Losses query sorts files; for the other folders the order cannot matter.
    order = ctx.rules['file_order'] if kind == 'line_losses' else 'name'
    return concat_files(ctx.site, kind, order, ctx)


def settings_filter(df, ctx, col_date):
    if MONTH:
        df = df[df[col_date].dt.strftime('%Y-%m') == MONTH]
    if MACHINES:
        df = df[df['MachineName'].isin(MACHINES)]
    return df.copy()


# ---------------- Plan_OPT ----------------
def P_Source(df, ctx):
    df = S_all_files(df, ctx, 'plan_opt')
    df['Date'] = parse_dt(df['YYYY-MM-DD'], '%Y-%m-%d')
    df['ShiftStartTime'] = parse_dt(df['ShiftStartTime'], '%d/%m/%Y %H:%M:%S')
    for c in ['Good Parts', 'Capacity', 'Downtime Units']:
        df[c] = num(df[c])
    for c in ['Total Time', 'Planned OpT', 'OEE Time']:
        df[c] = mins(df[c])
    return df


def P_Settings(df, ctx):
    return settings_filter(df, ctx, 'Date')


def P_Filtered_Rows1(df, ctx):
    df = df[df['MachineName'] != '']
    if ctx.site == 'Nottingham':
        df = df[df['MachineName'] != 'N11 Auto']
    return df.copy()


def P_Merged_Queries3_ShiftNaming(df, ctx):
    if ctx.rules['shift_file'] == 'none':
        ctx.note = 'skipped (rule shift_file=none)'
        return df
    sh = load_shift(ctx)
    return left_join(df, df['ShiftName'], sh, 'Shift Type', ['Shift'], ctx, 'Shift Naming')


def P_Filtered_Rows2_GT4(df, ctx):
    if ctx.site == 'Hull':
        df = df[df['MachineName'] != 'GT4 - Auto']
    return df.copy()


def month_key(d):
    return d.dt.to_period('M').dt.to_timestamp().dt.strftime('%d/%m/%Y')


def P_Merged_Queries_Targets(df, ctx):
    g = load_glide(ctx)
    for bucket, col in [('Unplanned Downtime (Total)', 'UPDT Target'),
                        ('Planned Downtime (Total)', 'PDT Target'), ('% OEE', 'OEE Target')]:
        lk = g[g['OEE Bucket'] == bucket].copy()
        lk['_key'] = lk['_mk'] + ' ' + lk['Attribute']
        lk = lk.rename(columns={'Value': col})
        key = machine_key(df, 'MachineName', ctx) + ' ' + month_key(df['Date'])
        df = left_join(df, key, lk, '_key', [col], ctx, f'Targets {bucket}')
    return df


def P_Filtered_Rows_EU(df, ctx):
    if ctx.rules['machine_filter'] != 'eu':
        ctx.note = 'skipped (rule machine_filter=none)'
        return df
    if ctx.site == 'Hull':
        return df[~df['MachineName'].isin(HULL_EU_EXCLUDE)].copy()
    keep = df['AreaName'].isin(NOTT_PLAN_AREAS) & ~df['MachineName'].isin(NOTT_PLAN_EXCLUDE)
    return df[keep].copy()


def P_Plan_OPT_Provideam_whole_minutes(df, ctx):
    if ctx.rules['minutes'] != 'whole':
        ctx.note = 'skipped (rule minutes=seconds)'
        return df
    for c in ['Total Time', 'Planned OpT', 'OEE Time']:
        df[c] = np.floor(df[c] + 1e-9)
    return df


PLAN_STEPS = [
    ('Source (all files in folder)', P_Source),
    ('Settings filter (month, machines)', P_Settings),
    ('Filtered Rows1 (machine not empty, N11 Auto)', P_Filtered_Rows1),
    ('Merged Queries3 (Shift Naming)', P_Merged_Queries3_ShiftNaming),
    ('Filtered Rows2 (GT4 - Auto)', P_Filtered_Rows2_GT4),
    ('Merged Queries (UPDT, PDT, OEE targets)', P_Merged_Queries_Targets),
    ('Filtered Rows (EU machine / area filter)', P_Filtered_Rows_EU),
    ('Plan OPT Provideam (times to whole minutes)', P_Plan_OPT_Provideam_whole_minutes),
]


# ---------------- Time_Utilisation ----------------
def T_Source(df, ctx):
    df = S_all_files(df, ctx, 'plan_opt')
    df['Date'] = parse_dt(df['YYYY-MM-DD'], '%Y-%m-%d')
    for c in ['Total Time', 'Planned OpT', 'OEE Time']:
        df[c] = mins(df[c])
    return df


def T_Filtered_Rows1(df, ctx):
    return df[df['MachineName'] != ''].copy()


def T_Grouped_Rows(df, ctx):
    g = df.groupby(['Date', 'MachineName'], dropna=False).agg(**{
        'OEE Time Total': ('OEE Time', lambda s: s.sum(min_count=1)),
        'Total Time Total': ('Total Time', lambda s: s.sum(min_count=1)),
        'Planned OPT Total': ('Planned OpT', lambda s: s.sum(min_count=1)),
    })
    return g.reset_index()


def T_Filtered_Rows2(df, ctx):
    return df[df['MachineName'] != SITE_MACHINE_OUT[ctx.site]].copy()


def T_Filtered_Rows3_EU(df, ctx):
    if ctx.rules['machine_filter'] != 'eu':
        ctx.note = 'skipped (rule machine_filter=none)'
        return df
    if ctx.site == 'Hull':
        return df[~df['MachineName'].isin(HULL_EU_EXCLUDE)].copy()
    return df[df['MachineName'].isin(NOTT_TU_KEEP)].copy()


TU_STEPS = [
    ('Source (all files in folder)', T_Source),
    ('Settings filter (month, machines)', P_Settings),
    ('Filtered Rows1 (machine not empty)', T_Filtered_Rows1),
    ('Grouped Rows (day x machine, minutes summed)', T_Grouped_Rows),
    ('Filtered Rows2 (GT4 - Auto / N11 Auto)', T_Filtered_Rows2),
    ('Filtered Rows3 (EU machine filter)', T_Filtered_Rows3_EU),
]


# ---------------- Daily_Losses ----------------
def L_Source(df, ctx):
    df = S_all_files(df, ctx, 'line_losses')
    df['StartTime'] = parse_dt(df['StartTime'], '%d/%m/%Y %H:%M:%S')
    df['ShiftStartDate'] = parse_dt(df['ShiftStartDate'], '%d/%m/%Y %H:%M:%S')
    df['Duration'] = mins(df['Duration'])
    return df


def L_Settings(df, ctx):
    df = df.copy()
    df['Date'] = df['StartTime'].dt.normalize()
    return settings_filter(df, ctx, 'Date')


def L_Replaced_Value_Breakdown(df, ctx):
    for a in ['Breakdown - Mechanical', 'Breakdown - Electrical']:
        df['OEELossLvl3SubGroup'] = df['OEELossLvl3SubGroup'].str.replace(
            a, 'Breakdown', regex=False)
    return df


def L_Filtered_Rows1(df, ctx):
    return df[df['MachineName'] != ''].copy()


def L_Replaced_Value_PlannedLoss(df, ctx):
    if ctx.site != 'Nottingham':
        ctx.note = 'skipped (Nottingham only)'
        return df
    for a, b in [('Planned Losses', 'Planned Loss'), ('Unplanned Losses', 'Unplanned Loss')]:
        df['OEELossLvl3Group'] = df['OEELossLvl3Group'].str.replace(a, b, regex=False)
    return df


def L_Removed_Duplicates(df, ctx):
    return df.drop_duplicates(['MachineName', 'StartTime'], keep='first').copy()


def L_Filtered_Rows2(df, ctx):
    keep = ((df['OEELossLvl3Group'] != '') & (df['OEELossLvl3Group'] != '0.Review Data')
            & ~df['MachineName'].isin(LOSS_MACHINES_OUT))
    return df[keep].copy()


def L_Merged_Queries4_ShiftNaming(df, ctx):
    if ctx.rules['shift_file'] == 'none':
        ctx.note = 'skipped (rule shift_file=none)'
        return df
    sh = load_shift(ctx)
    return left_join(df, df['ShiftName'], sh, 'Shift Type', ['Shift'], ctx, 'Shift Naming')


def L_Merged_Queries_VolumeLossCorrection(df, ctx):
    df = df.copy()
    df['ShiftStartDay'] = df['ShiftStartDate'].dt.normalize()
    key = text_date(df['ShiftStartDay']).str.cat(
        [df['MachineName'], df['OEELossLvl3Group'], df['OEELossLvl3SubGroup'],
         df['OEELossLvl3Desc']], sep=' ')
    return left_join(df, key, load_vlc(ctx), 'Volume Key', ['Loss Units', 'Time in Mins'],
                     ctx, 'Volume Loss Correction')


def L_Replaced_Value_after_VLC(df, ctx):
    pairs = [('Breaks and Meal', 'Meals & Breaks'), ('Lack of resources', 'Lack of Resource')]
    if ctx.site == 'Hull':
        pairs = [('Adjustments', 'Adjustment')] + pairs
    for a, b in pairs:
        df['OEELossLvl3SubGroup'] = df['OEELossLvl3SubGroup'].str.replace(a, b, regex=False)
    return df


def L_Merged_Queries1_EquipmentCoding(df, ctx):
    key = df['MachineName'] + ' ' + df['OEELossLvl3Desc']
    return left_join(df, key, load_equipment(ctx.site), 'Merged Unique Identifier',
                     ['Equipment'], ctx, 'Equipment Coding')


def L_Filtered_Rows3_groups(df, ctx):
    return df[df['OEELossLvl3Group'].isin(LOSS_GROUPS)].copy()


HULL_EU_RENAMES = [
    ('Adjustment', 'Adjustments'), ('Start-Up/End-Up & Sanitization', 'Start up and Sanitisation'),
    ('In Process Quality Control', 'In Process Quality'),
    ('Lack Of Resource', 'Lack Of Resources'), ('Meals & Breaks', 'Breaks / Meals'),
    ('Minor Stop', 'Minor Stops'), ('Other', 'Others'),
    ('Product Change Over', 'Product Changeover'),
    ('Start Up And Sanitisation', 'Start up and Sanitisation'),
    ('Lack Of Resourcess', 'Lack Of Resources'), ('Operater Asset Care', 'Operator Asset Care'),
    ('Beakdown - Mechanical', 'Breakdown'),
]
NOTT_RENAMES = [
    ('Minor Stop', 'Minor Stops'), ('Lack of Resources', 'Lack Of Resources'),
    ('Other', 'Others'), ('Meals & Breakss', 'Breaks / Meals'),
    ('In Process Quality Control', 'In Process Quality'),
    ('Start up & Sanitisation', 'Start up and Sanitisation'),
]


def L_Capitalized_and_sub_group_renames(df, ctx):
    if ctx.site == 'Nottingham':
        pairs, proper = NOTT_RENAMES, False
    elif ctx.rules['sub_group_names'] == 'eu':
        pairs, proper = HULL_EU_RENAMES, True
    else:
        ctx.note = 'skipped (rule sub_group_names=site: raw names)'
        return df
    col = df['OEELossLvl3SubGroup']
    if proper:
        col = col.str.title()
    for a, b in pairs:
        col = col.str.replace(a, b, regex=False)
    df['OEELossLvl3SubGroup'] = col
    return df


def L_Merged_Queries2_LossTargets(df, ctx):
    g = load_glide(ctx)
    g['_key'] = g['_mk'] + ' ' + g['OEE Bucket'] + ' ' + g['Attribute']
    start = df['ShiftStartDate'].dt.to_period('M').dt.to_timestamp().dt.strftime('%d/%m/%Y')
    key = (machine_key(df, 'MachineName', ctx) + ' ' + df['OEELossLvl3SubGroup']
           + ' ' + start.fillna(''))
    return left_join(df, key, g, '_key', ['Value'], ctx, 'Loss targets')


def L_Merged_Queries3_TargetsVolume(df, ctx):
    key = df['MachineName'] + ' ' + text_date(df['ShiftStartDay'])
    return left_join(df, key, load_targets_volume(ctx), 'Volume Key', ['Volume Total'], ctx,
                     'Targets Volume')


def L_Calculated_columns(df, ctx):
    df = df.copy()
    if ctx.rules['minutes'] == 'whole':
        df['Duration'] = np.floor(df['Duration'] + 1e-9)
    else:
        ctx.note = 'Duration keeps seconds'
    cu = df['Loss Units'] * df['Duration'] / df['Time in Mins']
    n_inf = int(np.isinf(cu).sum())
    if n_inf:
        ctx.warn(f'Loss in CUs: {n_inf} rows with Time in Mins = 0 (infinite), set to empty')
    df['Loss in CUs'] = cu.replace([np.inf, -np.inf], np.nan)
    return df


def L_Filtered_Rows_EU(df, ctx):
    if ctx.rules['machine_filter'] != 'eu':
        ctx.note = 'skipped (rule machine_filter=none)'
        return df
    out = HULL_EU_EXCLUDE if ctx.site == 'Hull' else NOTT_LOSS_EXCLUDE
    return df[~df['MachineName'].isin(out)].copy()


LOSS_STEPS = [
    ('Source (all files in folder)', L_Source),
    ('Settings filter (month, machines)', L_Settings),
    ('Replaced Value (Breakdown - Mechanical / Electrical)', L_Replaced_Value_Breakdown),
    ('Filtered Rows1 (machine not empty)', L_Filtered_Rows1),
    ('Replaced Value (Planned Losses to Planned Loss)', L_Replaced_Value_PlannedLoss),
    ('Removed Duplicates (machine + start time)', L_Removed_Duplicates),
    ('Filtered Rows2 (group, B1/B4/B5 Auto)', L_Filtered_Rows2),
    ('Merged Queries4 (Shift Naming)', L_Merged_Queries4_ShiftNaming),
    ('Merged Queries (Volume Loss Correction)', L_Merged_Queries_VolumeLossCorrection),
    ('Replaced Value (sub group names after VLC)', L_Replaced_Value_after_VLC),
    ('Merged Queries1 (Equipment Coding)', L_Merged_Queries1_EquipmentCoding),
    ('Filtered Rows3 (Not Logged, Planned, Unplanned)', L_Filtered_Rows3_groups),
    ('Capitalized Each Word + sub group renames', L_Capitalized_and_sub_group_renames),
    ('Merged Queries2 (loss targets, Value)', L_Merged_Queries2_LossTargets),
    ('Merged Queries3 (Targets Volume)', L_Merged_Queries3_TargetsVolume),
    ('Calculated columns (Duration, Loss in CUs)', L_Calculated_columns),
    ('Filtered Rows (EU machine filter)', L_Filtered_Rows_EU),
]

PIPES = {'Plan_OPT': PLAN_STEPS, 'Time_Utilisation': TU_STEPS, 'Daily_Losses': LOSS_STEPS}


def totals(df, table):
    out = {'rows': len(df)}
    for m in MEASURES[table]:
        out[m] = df[m].sum() if m in df.columns else np.nan
    return out


def run(table, site, rules, off=frozenset(), upto=None):
    """Run the M steps in order. Returns (final frame, trace, ctx)."""
    ctx = Ctx(site, rules)
    df, trace = None, []
    for name, fn in PIPES[table]:
        ctx.note = ''
        if name in off:
            ctx.note = 'switched off in settings'
        else:
            df = fn(df, ctx)
        trace.append({'step': name, 'note': ctx.note, **totals(df, table)})
        if upto == name:
            break
    return df, pd.DataFrame(trace).set_index('step'), ctx


def steps_of(table):
    return [n for n, _ in PIPES[table]]

## 5. Side by side: after each step, rows and sums

For each measure: EU, Site, and the gap (EU - Site). A step where the gap changes is a cause.

In [ ]:
def side_by_side(table, site):
    _, te, ce = run(table, site, RULES['EU'], OFF['EU'])
    _, ts, cs = run(table, site, RULES[site], OFF['SITE'])
    cols = ['rows'] + MEASURES[table]
    parts = {}
    for c in cols:
        parts[(c, 'EU')] = te[c]
        parts[(c, 'Site')] = ts[c]
        parts[(c, 'gap')] = te[c] - ts[c]
    sbs = pd.DataFrame(parts)
    gap = sbs.xs('gap', axis=1, level=1)
    changed = (gap.fillna(0).diff().abs() > TOL)
    changed.iloc[0] = (gap.fillna(0).iloc[0].abs() > TOL)
    split = changed.any(axis=1)
    return sbs, split, te, ts, ce, cs


def show_side_by_side(table, site):
    sbs, split, te, ts, ce, cs = side_by_side(table, site)
    print(f'===== {site} / {table}: EU rules vs {site} site rules =====')
    display(sbs.dropna(axis=1, how='all'))
    print('Steps where the gap changes (= a cause):')
    gap = sbs.xs('gap', axis=1, level=1).fillna(0)
    delta = gap.diff().fillna(gap.iloc[[0]])
    display(delta[split].loc[:, (delta[split].abs() > TOL).any()])
    notes = pd.concat([te['note'].rename('EU'), ts['note'].rename('Site')], axis=1)
    notes = notes[(notes['EU'] != '') | (notes['Site'] != '')]
    if len(notes):
        print('Step notes:')
        display(notes)
    for who, c in [('EU', ce), ('Site', cs)]:
        for w in c.warns:
            print(f'WARNING ({who}): {w}')
    return split


# Rows behind a step: multiset difference on the key columns
def rows_behind(table, site, step, by=BREAKDOWN_BY, top=TOP):
    eu, _, _ = run(table, site, RULES['EU'], OFF['EU'], upto=step)
    st, _, _ = run(table, site, RULES[site], OFF['SITE'], upto=step)
    return diff_frames(table, site, eu, st, by, top, 'EU', 'Site')


def diff_frames(table, site, a, b, by, top, na, nb):
    keys = KEYS[table]
    meas = [m for m in MEASURES[table] if m in a.columns and m in b.columns]
    ca = a.groupby(keys, dropna=False).size().rename('a')
    cb = b.groupby(keys, dropna=False).size().rename('b')
    j = pd.concat([ca, cb], axis=1).fillna(0)
    j['d'] = j['a'] - j['b']
    only_a = j[j['d'] > 0].reset_index()[keys]
    only_b = j[j['d'] < 0].reset_index()[keys]
    mn = load_machine_names(site)

    def enrich(f):
        f = f.merge(mn, on='MachineName', how='left')
        if 'Date' in f.columns:
            f['Date'] = f['Date'].dt.strftime('%Y-%m-%d')
        return f

    print(f'rows only in {na}: {int(j.loc[j.d > 0, "d"].sum())}  |  rows only in {nb}: '
          f'{int(-j.loc[j.d < 0, "d"].sum())}  |  keys with different counts: '
          f'{int((j.d != 0).sum())}')
    for label, keyset, frame in [(na, only_a, a), (nb, only_b, b)]:
        if not len(keyset):
            continue
        rows = enrich(frame.merge(keyset, on=keys, how='inner'))
        print(f'\n--- rows behind, only in {label} ---')
        for col in by:
            if col in rows.columns:
                grp = rows.groupby(col, dropna=False)
                g = grp[meas].sum()
                g.insert(0, 'rows', grp.size())
                print(f'by {col}')
                display(g.sort_values('rows', ascending=False).head(top))
    # same key, different measure values
    sa = a.groupby(keys, dropna=False)[meas].sum()
    sb = b.groupby(keys, dropna=False)[meas].sum()
    both = sa.join(sb, how='inner', lsuffix=f'_{na}', rsuffix=f'_{nb}')
    diffs = pd.DataFrame({m: both[f'{m}_{na}'] - both[f'{m}_{nb}'] for m in meas})
    diffs = diffs[(diffs.abs() > TOL).any(axis=1)]
    if len(diffs):
        print(f'\n--- same key, different values: {len(diffs)} keys ---')
        display(diffs.abs().sum(axis=1).sort_values(ascending=False).head(top).to_frame('abs diff')
                .join(diffs))
    return only_a, only_b

## 6. Same-rule check

The EU pipeline is run with the **site rules**. If every difference is explained, the sums match
the site report. What does not match is "unexplained" and listed with the rows behind it.

In [ ]:
def same_rule(table, site):
    st, _, _ = run(table, site, RULES[site], OFF['SITE'])
    eu, _, c = run(table, site, RULES[site], OFF['EU'])
    ts, te = totals(st, table), totals(eu, table)
    res = pd.DataFrame({'EU with site rules': te, 'Site': ts})
    res['gap'] = res.iloc[:, 0] - res.iloc[:, 1]
    res = res.dropna(how='all')
    ok = (res['gap'].fillna(0).abs() <= TOL).all()
    print(f'===== {site} / {table}: same-rule check: ' + ('MATCH' if ok else 'NOT EQUAL') + ' =====')
    display(res)
    if not ok:
        print('Unexplained, rows behind:')
        diff_frames(table, site, eu, st, BREAKDOWN_BY, TOP, 'EU-same-rules', 'Site')
    return ok, res

## 7. Discrepancy map

One row per cause (a rule). The effect is measured by starting from the **site** pipeline and
switching only that rule to the EU value. `interaction` is what is left of the total gap after
the single effects (rules that act together). `unexplained` is what the same-rule check could
not match.

In [ ]:
RULE_DOC = {
    'machine_filter': ('Filtered Rows (EU machine / area filter)', 'EU',
                       'EU report drops machines or areas the site report keeps (process, '
                       'manufacturing, S14, Dettol 1, Senokot Press ...).'),
    'minutes': ('Plan OPT Provideam / Calculated columns', 'EU',
                'EU cuts seconds off Duration and plan times (hours:minutes), the site keeps '
                'seconds.'),
    'sub_group_names': ('Capitalized Each Word + sub group renames', 'EU',
                        'EU renames Hull sub groups before the target key, so more or fewer '
                        'keys find a glidepath value.'),
    'target_key': ('Merged Queries (targets)', 'EU',
                   'EU key = machine-site, site key = machine; matches different glidepath '
                   'rows.'),
    'glidepath': ('Merged Queries (targets)', 'EU',
                  'EU reads the regional glidepath file, the site report its own file '
                  '(different cells, units).'),
    'shift_file': ('Merged Queries (Shift Naming)', 'EU',
                   'Different shift lookup file; rows change only if a shift name repeats in '
                   'the file.'),
    'file_order': ('Removed Duplicates', 'Site',
                   'Hull site sorts files by Date created before removing duplicates, so a '
                   'stop in two files keeps another copy.'),
}


def discrepancy_map(table, site):
    base, _, _ = run(table, site, RULES[site], OFF['SITE'])
    eu, _, _ = run(table, site, RULES['EU'], OFF['EU'])
    tb, te = pd.Series(totals(base, table)), pd.Series(totals(eu, table))
    gap = te - tb
    rows, single_sum = [], pd.Series(0.0, index=gap.index)
    for rule, (step, who, doc) in RULE_DOC.items():
        if RULES['EU'][rule] == RULES[site][rule]:
            continue
        r = dict(RULES[site])
        r[rule] = RULES['EU'][rule]
        d, _, _ = run(table, site, r, OFF['SITE'])
        eff = pd.Series(totals(d, table)) - tb
        if (eff.fillna(0).abs() <= TOL).all():
            continue
        single_sum = single_sum.add(eff.fillna(0), fill_value=0)
        rows.append({'site': site, 'table': table, 'cause (rule)': f'{rule}: '
                     f'{RULES[site][rule]} -> {RULES["EU"][rule]}', 'step': step,
                     'pipeline': who, 'rows affected': eff['rows'],
                     **{m: eff[m] for m in MEASURES[table]}, 'explained': doc})
    inter = (gap.fillna(0) - single_sum)
    if (inter.abs() > TOL).any():
        rows.append({'site': site, 'table': table, 'cause (rule)': 'interaction of rules',
                     'step': 'several', 'pipeline': 'both', 'rows affected': inter['rows'],
                     **{m: inter[m] for m in MEASURES[table]},
                     'explained': 'Rules acting on the same rows; the single effects do not add '
                                  'up to the total gap.'})
    eu_s, _, _ = run(table, site, RULES[site], OFF['EU'])
    un = pd.Series(totals(eu_s, table)) - tb
    if (un.fillna(0).abs() > TOL).any():
        rows.append({'site': site, 'table': table, 'cause (rule)': 'UNEXPLAINED',
                     'step': '?', 'pipeline': '?', 'rows affected': un['rows'],
                     **{m: un[m] for m in MEASURES[table]},
                     'explained': 'Still different with identical rules: see section 6.'})
    rows.append({'site': site, 'table': table, 'cause (rule)': 'TOTAL GAP (EU - Site)',
                 'step': '', 'pipeline': '', 'rows affected': gap['rows'],
                 **{m: gap[m] for m in MEASURES[table]}, 'explained': ''})
    return pd.DataFrame(rows)


def orphans(table, site):
    """Fact rows with no match in the EU-filtered Machine Names (they drop out of visuals that
    group by Dept / Area, but stay in a plain total)."""
    eu, _, _ = run(table, site, RULES['EU'], OFF['EU'])
    mn = load_machine_names(site)
    if site == 'Hull':
        ok = mn[mn['Dept'].isin(['Dries Pack', 'Liquids Pack'])]['MachineName']
    else:
        ok = mn[mn['Area'].isin(['SD Packing', 'Strepsils Packing'])]['MachineName']
    miss = eu[~eu['MachineName'].isin(ok)]
    t = pd.Series(totals(miss, table))
    print(f'{site} / {table}: EU rows with no match in the EU Machine Names: {len(miss)} '
          f'({miss["MachineName"].nunique()} machines)')
    return t

## 8. Run

In [ ]:
MAPS = []
SPLITS = {}
for site in SITES:
    for table in TABLES:
        SPLITS[(site, table)] = show_side_by_side(table, site)

### Rows behind the first split (per table and site)
Change `STEP` to look at any other step, or loop the list printed above.

In [ ]:
for (site, table), split in SPLITS.items():
    steps = [s for s, flag in split.items() if flag]
    if not steps:
        print(f'{site} / {table}: the sums never split')
        continue
    print(f'\n######## {site} / {table}: rows behind step "{steps[0]}"')
    rows_behind(table, site, steps[0])

In [ ]:
for site in SITES:
    for table in TABLES:
        same_rule(table, site)

In [ ]:
for site in SITES:
    for table in TABLES:
        MAPS.append(discrepancy_map(table, site))
MAP = pd.concat(MAPS, ignore_index=True)
display(MAP)

In [ ]:
for site in SITES:
    for table in TABLES:
        display(orphans(table, site).to_frame('EU rows without a Machine Names match'))